# 💳 Credit Card Fraud Alert System

This notebook implements the pipeline below end-to-end:

```
Customer makes transaction
          │
Transaction enters system
          │
System checks transaction
          │
Fraud Detection Model
          │
Risk Score
          │
   ┌──────┼───────┐
   ▼      ▼       ▼
 SAFE  SUSPICIOUS FRAUD
   │      │        │
 Allow  Review    Alert
```

**Tech stack:** Python, Jupyter, pandas, scikit-learn (Fraud Detection Model), matplotlib (visual alerts/dashboard).

> Note: This uses **synthetic transaction data** so the notebook runs standalone. Swap in a real dataset (e.g. a bank's transaction log or the Kaggle "Credit Card Fraud Detection" dataset) by replacing Step 1.

## Step 1 — Transaction enters the system
Simulate a stream of customer transactions with the features a real fraud model would use.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

np.random.seed(42)
plt.rcParams['figure.figsize'] = (8, 5)

n = 5000  # number of historical transactions

data = pd.DataFrame({
    'transaction_id': range(1, n + 1),
    'amount': np.random.exponential(scale=100, size=n).round(2),
    'hour': np.random.randint(0, 24, n),
    'distance_from_home': np.random.exponential(scale=20, size=n).round(2),
    'distance_from_last_transaction': np.random.exponential(scale=15, size=n).round(2),
    'ratio_to_median_purchase': np.random.exponential(scale=1.5, size=n).round(2),
    'repeat_retailer': np.random.choice([0, 1], size=n, p=[0.2, 0.8]),
    'used_chip': np.random.choice([0, 1], size=n, p=[0.3, 0.7]),
    'used_pin': np.random.choice([0, 1], size=n, p=[0.4, 0.6]),
    'online_order': np.random.choice([0, 1], size=n, p=[0.6, 0.4]),
})

# Ground-truth fraud label (only used to TRAIN the model; a real system would
# get this from confirmed chargebacks / customer disputes)
fraud_prob = (
    0.001
    + 0.15 * (data['amount'] > 300)
    + 0.20 * (data['distance_from_home'] > 50)
    + 0.15 * (data['ratio_to_median_purchase'] > 3)
    + 0.10 * (data['online_order'] == 1)
    - 0.10 * (data['used_chip'] == 1)
    - 0.10 * (data['used_pin'] == 1)
).clip(0, 1)

data['is_fraud'] = np.random.binomial(1, fraud_prob)

print(f"Total transactions: {len(data)}")
print(data['is_fraud'].value_counts().rename({0: 'Legitimate', 1: 'Fraud'}))
data.head()

## Step 2 — System checks / Fraud Detection Model
Train a classifier that acts as the **Fraud Detection Model** box in the flowchart. It outputs a fraud probability for every transaction.

In [ ]:
features = ['amount', 'hour', 'distance_from_home', 'distance_from_last_transaction',
            'ratio_to_median_purchase', 'repeat_retailer', 'used_chip', 'used_pin', 'online_order']

X = data[features]
y = data['is_fraud']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

model = RandomForestClassifier(
    n_estimators=200, max_depth=8, random_state=42, class_weight='balanced'
)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred, target_names=['Legitimate', 'Fraud']))

In [ ]:
# Confusion matrix -- how well the Fraud Detection Model separates the classes
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Legitimate', 'Fraud'])
disp.plot(cmap='Blues', values_format='d')
plt.title('Fraud Detection Model — Confusion Matrix')
plt.show()

In [ ]:
# Which features drive the model's decisions
importances = pd.Series(model.feature_importances_, index=features).sort_values()
importances.plot(kind='barh', color='#4C72B0')
plt.title('Feature Importance — Fraud Detection Model')
plt.xlabel('Importance')
plt.tight_layout()
plt.show()

## Step 3 — Risk Score
Convert the model's fraud probability into a 0–100 **Risk Score** for every transaction.

In [ ]:
data['risk_score'] = model.predict_proba(X)[:, 1] * 100
data[['transaction_id', 'amount', 'risk_score']].sort_values('risk_score', ascending=False).head(10)

## Step 4 — Branch: SAFE / SUSPICIOUS / FRAUD → Allow / Review / Alert
Apply thresholds to route each transaction, mirroring the flowchart's branch.

In [ ]:
RISK_THRESHOLDS = {
    'SAFE_MAX': 30,        # risk_score < 30  -> SAFE
    'SUSPICIOUS_MAX': 70,  # 30 <= score < 70 -> SUSPICIOUS, else FRAUD
}

ACTION_MAP = {
    'SAFE': 'Allow',
    'SUSPICIOUS': 'Review',
    'FRAUD': 'Alert',
}

def categorize(score: float) -> str:
    if score < RISK_THRESHOLDS['SAFE_MAX']:
        return 'SAFE'
    elif score < RISK_THRESHOLDS['SUSPICIOUS_MAX']:
        return 'SUSPICIOUS'
    else:
        return 'FRAUD'

data['risk_category'] = data['risk_score'].apply(categorize)
data['action'] = data['risk_category'].map(ACTION_MAP)

summary = data['risk_category'].value_counts().reindex(['SAFE', 'SUSPICIOUS', 'FRAUD'])
summary

## Dashboard — visualize the alert system

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

colors = {'SAFE': '#2ca02c', 'SUSPICIOUS': '#ff9800', 'FRAUD': '#d62728'}

# Risk score distribution by category
for cat in ['SAFE', 'SUSPICIOUS', 'FRAUD']:
    subset = data[data['risk_category'] == cat]['risk_score']
    axes[0].hist(subset, bins=30, alpha=0.7, label=cat, color=colors[cat])
axes[0].axvline(30, color='gray', linestyle='--', linewidth=1)
axes[0].axvline(70, color='gray', linestyle='--', linewidth=1)
axes[0].set_title('Risk Score Distribution')
axes[0].set_xlabel('Risk Score')
axes[0].set_ylabel('Number of Transactions')
axes[0].legend()

# Category breakdown
axes[1].pie(
    summary.values, labels=summary.index, autopct='%1.1f%%',
    colors=[colors[c] for c in summary.index], startangle=90
)
axes[1].set_title('Transactions by Risk Category')

plt.tight_layout()
plt.show()

In [ ]:
# Action volume — what the system actually does with each transaction
action_counts = data['action'].value_counts().reindex(['Allow', 'Review', 'Alert'])
action_colors = ['#2ca02c', '#ff9800', '#d62728']

action_counts.plot(kind='bar', color=action_colors)
plt.title('Actions Taken by the System')
plt.ylabel('Number of Transactions')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## Step 5 — Real-time transaction pipeline
This function is the whole flowchart as one callable: a single incoming transaction goes in, and an action comes out.

In [ ]:
def process_transaction(transaction: dict, verbose: bool = True):
    """Run one transaction through the full pipeline:
    Transaction -> System check -> Fraud Detection Model -> Risk Score -> Category -> Action
    """
    df_t = pd.DataFrame([transaction])[features]
    score = float(model.predict_proba(df_t)[:, 1][0] * 100)
    category = categorize(score)
    action = ACTION_MAP[category]

    if verbose:
        icon = {'Allow': '✅', 'Review': '🟠', 'Alert': '🚨'}[action]
        print(f"{icon}  Risk Score: {score:5.1f}/100  |  Category: {category:<10}  |  Action: {action}")

    return {'risk_score': round(score, 2), 'risk_category': category, 'action': action}


# Example: a suspicious late-night, far-from-home, high-ratio online purchase
sample_txn = {
    'amount': 850.0,
    'hour': 3,
    'distance_from_home': 120,
    'distance_from_last_transaction': 80,
    'ratio_to_median_purchase': 5.2,
    'repeat_retailer': 0,
    'used_chip': 0,
    'used_pin': 0,
    'online_order': 1,
}

result = process_transaction(sample_txn)
result

In [ ]:
# A normal, low-risk transaction for comparison
normal_txn = {
    'amount': 25.0,
    'hour': 14,
    'distance_from_home': 2,
    'distance_from_last_transaction': 1,
    'ratio_to_median_purchase': 0.8,
    'repeat_retailer': 1,
    'used_chip': 1,
    'used_pin': 1,
    'online_order': 0,
}

process_transaction(normal_txn);

## Notes for production use
- Replace the synthetic dataset in **Step 1** with real transaction/chargeback history.
- Retrain the model periodically as fraud patterns shift (concept drift).
- Tune `RISK_THRESHOLDS` against business cost: false Alerts annoy customers, false Allows cost money.
- `SUSPICIOUS` → `Review` transactions should route to a human analyst queue or a step-up authentication (OTP) flow, not silently pass through.
- Log every `Alert` for compliance/audit trail.